<h1 style="color: black;">Statistical estimation of seasonality</h1>

<hr>

<span style="font-size:25px; color:black;"><b>Overview</b></span>  
<span style="font-size:16px; color:black; display:block; margin-top:10px;">
In this notebook, we estimate seasonal distribution of extreme events for any location in the world. 
We explore when the most extreme events cluster in time, the strength of their timing, 
and their cumulative effect in defining the length of the identified seasons.
</span>

<span style="font-size:25px; color:black; display:block; margin-top:20px;"><b>Method</b></span>
<ol style="font-size:16px; color:black; margin-top:5px;">
  <li>Correct for trends in the time series using the Butterworth low-pass filter;</li>
  <li>Extract the number of required extreme events (peaks) over a user-defined threshold;</li>
  <li>Assess the seasonality of selected peaks based on the mixture model of the Von Mises–Fisher distribution;</li>
  <li>Determine the length of the identified seasons from the first derivative of the model's distribution function.</li>
</ol>


---

<span style="font-size:25px; color:black"><b>Key notes</b></span>

<ul style="font-size:16px; color:black">
  <li><b>Point 1:</b> The butter worth low pass filter is sensitive to nan values. ensure all nan values are dropped</li>
  <li><b>Point 2:</b> The selection of peaks is over the entire timeseries and should be modified as required</li>
  <li><b>Point 3:</b> Minimum number of angles to fit the Von Mises-Fisher distribution correctly is 30</li>
</ul>

<span style="font-size:25px; color:black"><b>Key papers</b></span>
<ul style="font-size:16px; color:black">
  <li><b>ButterWorth low-pass filter:</b> Roberts, J. and Roberts, T.D. (1978) ‘Use of the Butterworth low-pass filter for oceanographic data’, Journal of Geophysical Research: Oceans, 83(C11), pp. 5510–5514. Available at: https://doi.org/10.1029/JC083iC11p05510.</li>
    
  <li><b>Extreme values modelling:</b> Coles, S. (2001), An Introduction to Statistical Modeling of Extreme Values, Springer, London, U. K.</li>
  
  <li><b> Von Mises-Fisher distribution :</b> Hornik, K. and Grün, B. (2014) ‘movMF: An R Package for Fitting Mixtures of von Mises-Fisher Distributions’, Journal of Statistical Software, 58, pp. 1–31. Available at: https://doi.org/10.18637/jss.v058.i10.</li>
</ul>

<span style="font-size:25px; color:black"><b>Key requirement</b></span>

<ul style="font-size:16px;color:black">
  <li><b>This notebook requires embedding an R interpreter in Python using <code>rpy2</code>.</b></li>

  <li><b>Step 1:</b> Install R on your system:
    <ul style="font-size:16px; color:black">
      <li><b>Windows:</b> Download and install from <a href="https://cran.r-project.org/bin/windows/base/">CRAN</a>.</li>
      <li><b>macOS:</b> Download from <a href="https://cran.r-project.org/bin/macosx/">CRAN</a>.</li>
      <li><b>Linux:</b> Use your package manager to install R.</li>
    </ul>
  </li>

  <li><b>Step 2:</b> Install <code>rpy2</code> in Python:
    <pre><code># Using pip
pip install rpy2
</code></pre>
  </li>

  <li><b>Step 3:</b> Install the required R packages.
    <ul style="font-size:16px; color:black">
      <li>
        <b>Option 1:</b> From R console:
        <pre style="margin-top:5px; margin-bottom:10px;"><code>install.packages(c("movMF", "circular", "CircStats", "ggplot2"))</code></pre>
      </li>
      <li>
        <b>Option 2:</b> From Python using rpy2:
        <pre style="margin-top:5px;"><code>import rpy2.robjects.packages as rpackages
utils = rpackages.importr('utils')
utils.chooseCRANmirror(ind=1)  
packages_to_install = ['movMF', 'circular', 'CircStats', 'ggplot2']
for pkg in packages_to_install:
    if not rpackages.isinstalled(pkg):
        utils.install_packages(pkg)</code></pre>
      </li>
    </ul>
  </li>
</ul>

In [ ]:
# Import necessary python packages
import glob
import xarray as xr
import pandas as pd
import numpy as np
import math
import os
from scipy.signal import butter, filtfilt

<span style="font-size:36px; font-weight:bold; color:black;">Step 0: Access the input data and select a location of choice</span>

In [ ]:
def extract_number(filename):
    return int(''.join(filter(str.isdigit, filename)))

station_files = glob.glob('/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/*.parquet')
station_files.sort(key=extract_number)
print(station_files)

['/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_0.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_1.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_2.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_3.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_4.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_5.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_6.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_7.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_8.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_9.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_10.parquet', '/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/station_11.parquet', '/home/aapolo

In [ ]:
one_station = station_files[1827] 
input_station = pd.read_parquet(one_station)
input_station = input_station [['surge_time','skew_surge']]
input_station

,surge_time,skew_surge
0,1950-01-01 11:30:00,0.176
1,1950-01-02 11:00:00,0.279
2,1950-01-02 11:00:00,0.303
3,1950-01-03 14:40:00,0.205
4,1950-01-04 11:00:00,0.102
...,...,...
43654,2024-12-29 06:40:00,0.195
43655,2024-12-29 14:00:00,0.215
43656,2024-12-31 05:10:00,0.244
43657,2024-12-31 11:20:00,0.246


<span style="font-size:36px; font-weight:bold; color:black;">Step 1: Filter the timeseries</span>

In [ ]:
def butterworth_low_pass_filter(data: pd.Series, cutoff: float = 0.0000000316880878140, order: int = 2, sample_rate: float = 1/3600) -> pd.Series:
    nyq = 0.5 * sample_rate
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered = filtfilt(b, a, input_station.skew_surge)
    cleaned_data = input_station.skew_surge - filtered
    filtered_surge = pd.DataFrame({
        'date': input_station['surge_time'],
        'filtered_surge': cleaned_data
    })
    return filtered_surge


<span style="font-size:36px; font-weight:bold; color:black;">Step 2: Decluster the timeseries</span>

In [ ]:
def decluster_timeseries(filtered_data: pd.DataFrame, indep: int = 36, quantile: float = 0.80) -> pd.DataFrame:
    filtered_data['date'] = pd.to_datetime(filtered_data['date'])
    threshold = filtered_data['filtered_surge'].quantile(quantile)
    peaks = filtered_data[filtered_data['filtered_surge'] > threshold].copy()
    peaks = peaks.sort_values(by='date').reset_index(drop=True)

    selected_peaks = []
    prev_peak = None

    for _, peak in peaks.iterrows():
        peak_time = peak['date']
        peak_value = peak['filtered_surge']
        
        window_start = peak_time - pd.Timedelta(hours=indep)
        window_end = peak_time + pd.Timedelta(hours=indep)

        if prev_peak is not None and prev_peak['window_end'] >= window_start:
            # Overlapping: keep the higher of the two
            if peak_value > prev_peak['peak']['filtered_surge']:
                prev_peak = {'window_start': window_start, 'window_end': window_end, 'peak': peak}
            else:
                continue
        else:
            # No overlap or first peak
            if prev_peak is not None:
                selected_peaks.append(prev_peak['peak'])
            prev_peak = {'window_start': window_start, 'window_end': window_end, 'peak': peak}

    # Add the last selected peak if it exists
    if prev_peak is not None:
        selected_peaks.append(prev_peak['peak'])

    # Create a new DataFrame of the selected peaks
    declustered_peaks_df = pd.DataFrame(selected_peaks).reset_index(drop=True)
    
    return declustered_peaks_df

<span style="font-size:36px; font-weight:bold; color:black;">Step 3: Extract Peaks</span>

In [ ]:
def extract_top_peaks(selected_peaks: pd.DataFrame, top_n: int = 117) -> pd.DataFrame:
    sort_df = selected_peaks.sort_values(by='filtered_surge', ascending=False)
    Top_117 = sort_df[:top_n]
    Top_117 = Top_117.reset_index(drop=True)
    Top_117['doy'] = Top_117.date.dt.dayofyear
    Top_117['doy radian'] = Top_117['doy'] / 365 * (2 * math.pi)
    
    return Top_117

<span style="font-size:36px; font-weight:bold; color:black;">Step 4: Extract day of year in radians</span>

In [ ]:
def extract_doy_radians(df: pd.DataFrame) -> np.ndarray:
    return df['doy radian'].to_numpy()

<span style="font-size:36px; font-weight:bold; color:black;">Step 5: Call the functions</span>

In [ ]:
filtered_data = butterworth_low_pass_filter(input_station)
print ('Filtered_Data:\n', filtered_data)
selected_peaks = decluster_timeseries(filtered_data)
print ('Selected_Peaks:\n',selected_peaks)
top_peaks_df = extract_top_peaks(selected_peaks)
print ('Top_peaks:\n',top_peaks_df)
angles = extract_doy_radians(top_peaks_df)
print ('Angles:\n',angles)

Filtered_Data:
                      date  filtered_surge
0     1950-01-01 11:30:00        0.017134
1     1950-01-02 11:00:00        0.120189
2     1950-01-02 11:00:00        0.144245
3     1950-01-03 14:40:00        0.046300
4     1950-01-04 11:00:00       -0.056645
...                   ...             ...
43654 2024-12-29 06:40:00        0.127575
43655 2024-12-29 14:00:00        0.147575
43656 2024-12-31 05:10:00        0.176575
43657 2024-12-31 11:20:00        0.178575
43658 2024-12-31 23:50:00        0.192575

[43659 rows x 2 columns]
Selected_Peaks:
                     date  filtered_surge
0    1950-01-02 11:00:00        0.144245
1    1950-02-19 12:30:00        0.141279
2    1950-02-24 04:10:00        0.197886
3    1950-03-07 00:10:00        0.130988
4    1950-03-11 21:40:00        0.150319
...                  ...             ...
1703 2024-11-25 02:10:00        0.302575
1704 2024-12-02 17:20:00        0.265575
1705 2024-12-22 05:50:00        0.276575
1706 2024-12-25 09:20:00   

<span style="font-size:36px; font-weight:bold; color:black;">Step 6: Embed R in Python</span>

In [ ]:
os.environ['R_HOME'] = '/sw/arch/RHEL8/EB_production/2023/software/R/4.3.2-gfbf-2023a/lib64/R'
import rpy2.robjects as robjects
from rpy2.robjects.packages import importr
from rpy2.robjects import r
movMF = importr('movMF')
circular = importr('circular')
CircStats = importr('CircStats')

<span style="font-size:36px; font-weight:bold; color:black;">Step 7: Null Hypothesis test of circular uniformity</span>

In [ ]:
robjects.globalenv['angles'] = robjects.FloatVector(angles)

In [ ]:
r('''
# Circular data in radians
angles_circular <- circular(angles, units = "radians") 

#Conduct the null hypothesis test of circular uniformity for the data
# 1. Rayleigh's Test
rayleigh_test <- rayleigh.test(angles_circular)
print(rayleigh_test)

# 2. Rao's Spacing Test
rao_test <- rao.spacing.test(angles_circular)
print(rao_test)

# 3. Kuiper's Test
kuiper_test <- kuiper.test(angles_circular)
print(kuiper_test)

# 4. Watson's Test
watson_test <- watson.test(angles_circular)
print(watson_test)

#Only the Rayleigh test returns a list with that contains the P-value even though they all print the P-value to the screen.
#Therefore to extract the P-value of the other three tests, we need to modify the original function of the three other tests

# 1. RAO SPACING 
rao2 <- function (x, alpha = 0, rad = TRUE) 
{
  rao.table <- NULL
  data(rao.table, package = "CircStats", envir = sys.frame(which = sys.nframe()))
  if (rad == TRUE) 
    x <- deg(x)
  x <- sort(x %% 360)
  n <- length(x)
  spacings <- c(diff(x), x[1] - x[n] + 360)
  U <- 1/2 * sum(abs(spacings - 360/n))
  if (n < 4) 
    stop("Sample size too small")
  if (n <= 30) 
    table.row <- n - 3
  else if (n <= 32) 
    table.row <- 27
  else if (n <= 37) 
    table.row <- 28
  else if (n <= 42) 
    table.row <- 29
  else if (n <= 47) 
    table.row <- 30
  else if (n <= 62) 
    table.row <- 31
  else if (n <= 87) 
    table.row <- 32
  else if (n <= 125) 
    table.row <- 33
  else if (n <= 175) 
    table.row <- 34
  else if (n <= 250) 
    table.row <- 35
  else if (n <= 350) 
    table.row <- 36
  else if (n <= 450) 
    table.row <- 37
  else if (n <= 550) 
    table.row <- 38
  else if (n <= 650) 
    table.row <- 39
  else if (n <= 750) 
    table.row <- 40
  else if (n <= 850) 
    table.row <- 41
  else if (n <= 950) 
    table.row <- 42
  else table.row <- 43
  
  numeric_p_value <- NA  # Initialize numeric p_value
  
  if (alpha == 0) {
    cat("\n")
    cat("       Rao's Spacing Test of Uniformity", "\n", "\n")
    cat("Test Statistic =", round(U, 5), "\n")
    if (U > rao.table[table.row, 1]) {
      cat("P-value < 0.001", "\n", "\n")
      numeric_p_value <- 0.001  # Set p-value as 0.001 for "< 0.001"
    } else if (U > rao.table[table.row, 2]) {
      cat("0.001 < P-value < 0.01", "\n", "\n")
      numeric_p_value <- 0.01  # Set p-value as 0.01 for "0.001 < P-value < 0.01"
    } else if (U > rao.table[table.row, 3]) {
      cat("0.01 < P-value < 0.05", "\n", "\n")
      numeric_p_value <- 0.05  # Set p-value as 0.05 for "0.01 < P-value < 0.05"
    } else if (U > rao.table[table.row, 4]) {
      cat("0.05 < P-value < 0.10", "\n", "\n")
      numeric_p_value <- 0.10  # Set p-value as 0.10 for "0.05 < P-value < 0.10"
    } else {
      cat("P-value > 0.10", "\n", "\n")
      numeric_p_value <- 1  # Set p-value as 1 for "> 0.10"
    }
  }
  else {
    cat("\n")
    cat("       Rao's Spacing Test of Uniformity", "\n", "\n")
    cat("Test Statistic =", round(U, 5), "\n")
    if (sum(alpha == c(0.001, 0.01, 0.05, 0.1)) == 0) 
      stop("Invalid significance level")
    table.col <- (1:4)[alpha == c(0.001, 0.01, 0.05, 0.1)]
    critical <- rao.table[table.row, table.col]
    cat("Level", alpha, "critical value =", critical, "\n")
    if (U > critical) 
      cat("Reject null hypothesis of uniformity", "\n", "\n")
    else cat("Do not reject null hypothesis of uniformity", "\n", "\n")
  }
  
  return(list(Test_Statistic = U, P_Value = numeric_p_value))  # Return both the test statistic and numeric p-value
}

# Usage of the modified RAO spacing function:
rao_result <- rao2(angles_circular)
print(rao_result$Test_Statistic)
print(rao_result$P_Value)

# 2. Kuiper_test
kuiper_test <- function(x, alpha = 0) {
  cat("\n", "      Kuiper's Test of Uniformity", "\n", "\n")
  
  # Critical values for Kuiper's test
  kuiper_crits <- cbind(c(0.15, 0.1, 0.05, 0.025, 0.01), c(1.537, 1.62, 1.747, 1.862, 2.001))
  
  # Sort the input and transform it
  x <- sort(x %% (2 * pi))/(2 * pi)
  n <- length(x)
  i <- c(1:n)
  
  # Compute the test statistic
  D.P <- max(i/n - x)
  D.M <- max(x - (i - 1)/n)
  V <- (D.P + D.M) * (sqrt(n) + 0.155 + 0.24/sqrt(n))
  cat("Test Statistic:", round(V, 4), "\n")
  
  # Initialize p-value variable
  p_value <- NA
  
  if (alpha == 0) {
    # Assign numeric p-value based on the value of the test statistic
    if (V < 1.537) {
      p_value <- 0.15
    } else if (V < 1.62) {
      p_value <- 0.125
    } else if (V < 1.747) {
      p_value <- 0.075
    } else if (V < 1.862) {
      p_value <- 0.0375
    } else if (V < 2.001) {
      p_value <- 0.0175
    } else {
      p_value <- 0.01
    }
    cat("P-value:", p_value, "\n", "\n")
  } else {
    # Critical value comparison if alpha is provided
    Critical <- kuiper_crits[(1:5)[alpha == c(kuiper_crits[, 1])], 2]
    cat("Level", alpha, "Critical Value:", round(Critical, 4), "\n")
    
    # Decision on the null hypothesis
    if (V > Critical) {
      cat("Reject Null Hypothesis", "\n", "\n")
    } else {
      cat("Do Not Reject Null Hypothesis", "\n", "\n")
    }
  }
  
  # Return both the test statistic and the p-value
  return(list(test_statistic = V, p_value = p_value))
}

# Usage of the modified Kuiper test function:
kuiper_result <- kuiper_test(angles_circular)
print(kuiper_result$test_statistic)  
print(kuiper_result$p_value)         

# 3. Watson test
watson <- function(x, alpha = 0, dist = "uniform") {
  # Initialize p-value variable
  p_value <- NA
  
  if(dist == "uniform") {
    cat("\n", "      Watson's Test for Circular Uniformity", "\n", "\n")
    n <- length(x)
    u <- sort(x)/(2 * pi)
    u.bar <- mean(u)
    i <- seq(1:n)
    sum.terms <- (u - u.bar - (2 * i - 1)/(2 * n) + 0.5)^2
    u2 <- sum(sum.terms) + 1/(12 * n)
    u2 <- (u2 - 0.1/n + 0.1/(n^2)) * (1 + 0.8/n)
    crits <- c(99, 0.267, 0.221, 0.187, 0.152)
    
    if(n < 8) {
      cat("Total Sample Size < 8:  Results are not valid", "\n", "\n")
      return(list(test_statistic = u2, p_value = NA))
    }
    
    cat("Test Statistic:", round(u2, 4), "\n")
    
    if(sum(alpha == c(0, 0.01, 0.025, 0.05, 0.1)) == 0)
      stop("Invalid input for alpha")
    
    if(alpha == 0) {
      if(u2 > 0.267) {
        p_value <- 0.01
      } else if(u2 > 0.221) {
        p_value <- 0.0175
      } else if(u2 > 0.187) {
        p_value <- 0.0375
      } else if(u2 > 0.152) {
        p_value <- 0.1
      } else {
        p_value <- 0.15
      }
      cat("Approximate P-value:", p_value, "\n", "\n")
    } else {
      index <- (1:5)[alpha == c(0, 0.01, 0.025, 0.05, 0.1)]
      Critical <- crits[index]
      if(u2 > Critical)
        Reject <- "Reject Null Hypothesis"
      else
        Reject <- "Do Not Reject Null Hypothesis"
      
      cat("Level", alpha, "Critical Value:", round(Critical, 4), "\n")
      cat(Reject, "\n", "\n")
    }
    
  } else if(dist == "vm") {
    cat("\n", "      Watson's Test for the von Mises Distribution", "\n", "\n")
    u2.crits <- cbind(c(0, 0.5, 1, 1.5, 2, 4, 100),
                      c(0.052, 0.056, 0.066, 0.077, 0.084, 0.093, 0.096), c(0.061, 0.066, 0.079, 0.092, 0.101, 0.113, 0.117), 
                      c(0.081, 0.09, 0.11, 0.128, 0.142, 0.158, 0.164))
    n <- length(x)
    mu.hat <- circ.mean(x)
    kappa.hat <- est.kappa(x)
    x <- (x - mu.hat) %% (2 * pi)
    x <- matrix(x, ncol = 1)
    z <- apply(x, 1, pvm, 0, kappa.hat)
    z <- sort(z)
    z.bar <- mean(z)
    i <- c(1:n)
    sum.terms <- (z - (2 * i - 1)/(2 * n))^2
    Value <- sum(sum.terms) - n * (z.bar - 0.5)^2 + 1/(12 * n)
    
    if(kappa.hat < 0.25)
      row <- 1
    else if(kappa.hat < 0.75)
      row <- 2
    else if(kappa.hat < 1.25)
      row <- 3
    else if(kappa.hat < 1.75)
      row <- 4
    else if(kappa.hat < 3)
      row <- 5
    else if(kappa.hat < 5)
      row <- 6
    else
      row <- 7
    
    if(alpha != 0) {
      if(alpha == 0.1)
        col <- 2
      else if(alpha == 0.05)
        col <- 3
      else if(alpha == 0.01)
        col <- 4
      else {
        stop("Invalid input for alpha", "\n", "\n")
      }
      Critical <- u2.crits[row, col]
      if(Value > Critical)
        Reject <- "Reject Null Hypothesis"
      else
        Reject <- "Do Not Reject Null Hypothesis"
      
      cat("Test Statistic:", round(Value, 4), "\n")
      cat("Level", alpha, "Critical Value:", round(Critical, 4), "\n")
      cat(Reject, "\n", "\n")
      
    } else {
      cat("Test Statistic:", round(Value, 4), "\n")
      if(Value < u2.crits[row, 2])
        p_value <- 0.1
      else if((Value >= u2.crits[row, 2]) && (Value < u2.crits[row, 3]))
        p_value <- 0.075
      else if((Value >= u2.crits[row, 3]) && (Value < u2.crits[row, 4]))
        p_value <- 0.025
      else
        p_value <- 0.01
      cat("Approximate P-value:", p_value, "\n", "\n")
    }
    
  } else
    stop("Distribution must be either uniform or von Mises")
  
  # Return both the test statistic and the numeric p-value
  return(list(test_statistic = ifelse(dist == "uniform", round(u2, 4), round(Value, 4)), p_value = p_value))
}

# Usage of the modified Watson test function:
watson_result <- watson(angles_circular)
print(watson_result$test_statistic)  
print(watson_result$p_value)         

# Combine p-values into a vector
p_values <- c(rayleigh_test$p.value, kuiper_result$p_value, watson_result$p_value, rao_result$P_Value)
p_values
''')


       Rayleigh Test of Uniformity 
       General Unimodal Alternative 

Test Statistic:  0.7813 
P-value:  0 


       Rao's Spacing Test of Uniformity 
 
Test Statistic = 209.8883 
P-value < 0.001 
 

       Kuiper's Test of Uniformity 
 
Test Statistic:  6.2735 
P-value < 0.01 
 

       Watson's Test for Circular Uniformity 
 
Test Statistic: 3.8445 
P-value < 0.01 
 

       Rao's Spacing Test of Uniformity 
 
Test Statistic = 209.8883 
P-value < 0.001 
 
[1] 209.8883
[1] 0.001

       Kuiper's Test of Uniformity 
 
Test Statistic: 6.2735 
P-value: 0.01 
 
[1] 6.273479
[1] 0.01

       Watson's Test for Circular Uniformity 
 
Test Statistic: 3.85 
Approximate P-value: 0.01 
 
[1] 3.85
[1] 0.01


0.000000,0.010000,0.010000,0.001000


<span style="font-size:36px; font-weight:bold; color:black;">Step 8: Compute Bonferroni correction</span>

In [ ]:
r('''
alpha <- 0.05
alpha_adjusted <- alpha / 4  # For four tests
Bonferronicor <- data.frame(
    Test = c("Rayleigh", "Kuiper", "Watson", "Rao"),
    P_Value = p_values,
    Significant = p_values < alpha_adjusted
)
''')

Test,P_Value,Significant
...,...,...


In [ ]:
bonferroni_df = r('Bonferronicor')
print(bonferroni_df)

      Test      P_Value Significant
1 Rayleigh 9.660513e-32        TRUE
2   Kuiper 1.000000e-02        TRUE
3   Watson 1.000000e-02        TRUE
4      Rao 1.000000e-03        TRUE



<span style="font-size:36px; font-weight:bold; color:black;">Step 9: Fit the Von Mises-Fisher Model</span>

In [ ]:
r('''
# Check if all tests are significant and update the list
if (all(Bonferronicor$Significant)) {
  uniformitytest <- 1
} else if (any(!Bonferronicor$Significant)) {
  uniformitytest <- 0
}

# If the uniformity test is 0, stop the process
if (uniformitytest == 0) {
  stop()
} else {
  set.seed(123)

  # Convert angles to Cartesian coordinates
  prepare_data <- function(angles) {
    cbind(cos(angles), sin(angles))
  }
  
  data_matrix <- prepare_data(angles)

  # Fit von Mises model and compute BIC
  fit_von_mises <- function(k, data) {
    fit <- movMF(data, k = k, control = list(E = "hardmax", nruns = 20))
    BIC_value <- BIC(fit)
    return(list(fit = fit, BIC = BIC_value))
  }

  # Extract the best model by BIC
  get_best_model <- function(models) {
    BIC_values <- sapply(models, function(res) res$BIC)
    best_model_index <- which.min(BIC_values)
    list(model = models[[best_model_index]]$fit, index = best_model_index)
  }

  # Fit von Mises mixtures from 1 to 3 components
  von_mises_mixture <- lapply(1:3, function(K) fit_von_mises(K, data_matrix))

  # Extract BIC values and find the best model (lowest BIC)
  BIC_values <- sapply(von_mises_mixture, function(res) res$BIC)
  best_model_index <- which.min(BIC_values)
  best_model <- von_mises_mixture[[best_model_index]]$fit
  
  # Extract fitted parameters from the best model
  mean_directions <- best_model$theta   # mean_direction
  mixing_proportion <- best_model$alpha   # mixing proportion
  print(mixing_proportion)
  # Extract the estimated mean directions (in Cartesian form)
  model_parameters <- print(best_model)
  
  # Create empirical CDF for observed data
  observed_angles <- sort(as.numeric(angles_circular))
  observed_probs <- ecdf(observed_angles)(observed_angles)

  # Simulate model-based angles for Q-Q plot and match quantiles
  set.seed(123)
  sim_data <- rmovMF(1000, mean_directions, mixing_proportion)
  sim_angles <- atan2(sim_data[, 2], sim_data[, 1]) %% (2 * pi)
  sim_angles <- sort(sim_angles)
  sim_probs <- ecdf(sim_angles)(sim_angles)

  # Match quantiles
  lookup_table <- data.frame(sim_probs, sim_angles)
  matched_quantiles <- sapply(observed_probs, function(p) {
      idx <- which.min(abs(lookup_table$sim_probs - p))
      lookup_table$sim_angles[idx]
    })

  # Calculate circular correlation
  calculate_circular_correlation <- function(obs_ang, mod_ang) {
    obs_circ <- circular(obs_ang, type = "angles", units = "radians")
    mod_circ <- circular( mod_ang, type = "angles", units = "radians")
    circular_correlation <- cor.circular(obs_circ, mod_circ)
  }

  circular_correlation <- calculate_circular_correlation(observed_angles, matched_quantiles)
  print(paste("Circular Correlation Coefficient:", circular_correlation))

  # Convert Cartesian mean directions to degrees
  convert_cartesian_to_angles <- function(directions) {
    angles <- atan2(directions[, 2], directions[, 1])
    deg <- angles * (180 / pi)
    deg[deg < 0] <- deg[deg < 0] + 360
    list(radians = angles, degrees = deg)
  }
  
  conversion <- convert_cartesian_to_angles(mean_directions)
  print("Estimated mean directions (in radians):")
  print(conversion$radians)
  print("Mean directions in degrees (normalized):")
  print(conversion$degrees)

  # Find local maxima and minima of the first derivative
  find_local_extrema <- function(derivatives) {
    maxima <- which(diff(sign(diff(derivatives))) == -2) + 1
    minima <- which(diff(sign(diff(derivatives))) == 2) + 1
    list(maxima = maxima, minima = minima)
  }
  
  # Compute density and its first numerical derivative from the model
  compute_density_and_derivative <- function(directions, proportion) {
    angles_seq <- seq(0, 2 * pi, length.out = 5000)
    density_values <- dmovMF(cbind(cos(angles_seq), sin(angles_seq)), directions, proportion)
    density_diff <- diff(density_values) / diff(angles_seq)
    density_diff <- c(NA, density_diff)  # Align length with angles_seq
    list(angles = angles_seq, density = density_values, derivative = density_diff)
  }
  
  density_results <- compute_density_and_derivative(mean_directions, mixing_proportion)
  angles_seq <- density_results$angles
  density_values <- density_results$density
  density_diff <- density_results$derivative
    
  extrema <- find_local_extrema(density_diff)

  local_maxima_indices <- extrema$maxima
  local_minima_indices <- extrema$minima

  maxima_info <- data.frame(
    angle = angles_seq[local_maxima_indices] * (180 / pi),
    density = density_values[local_maxima_indices],
    first_derivative = density_diff[local_maxima_indices]
  )

  minima_info <- data.frame(
    angle = angles_seq[local_minima_indices] * (180 / pi),
    density = density_values[local_minima_indices],
    first_derivative = density_diff[local_minima_indices]
  )

  print(maxima_info)
  print(minima_info)
  
  export_component_variables <- function(w,
                                      primary_mean = NA, secondary_mean = NA, tertiary_mean = NA,
                                      primary_angle_difference = NA, secondary_angle_difference = NA, tertiary_angle_difference = NA,
                                      rsquared_value = NA, primary_mix = NA, secondary_mix = NA, tertiary_mix = NA,
                                      primary_mean_season_start = NA, primary_mean_season_end = NA,
                                      secondary_mean_season_start = NA, secondary_mean_season_end = NA,
                                      tertiary_mean_season_start = NA, tertiary_mean_season_end = NA,
                                      count_within_range1 = NA, count_within_range2 = NA, count_within_range3 = NA,
                                      circ_correlation = NA) {
  assign("w", w, envir = .GlobalEnv)
  assign("rsquared_value", rsquared_value, envir = .GlobalEnv)
  assign("primary_mean", primary_mean, envir = .GlobalEnv)
  assign("primary_angle_difference", primary_angle_difference, envir = .GlobalEnv)
  assign("primary_mix", primary_mix, envir = .GlobalEnv)
  assign("primary_mean_season_start", primary_mean_season_start, envir = .GlobalEnv)
  assign("primary_mean_season_end", primary_mean_season_end, envir = .GlobalEnv)
  assign("count_within_range1", count_within_range1, envir = .GlobalEnv)
  assign("circ_correlation", circ_correlation, envir = .GlobalEnv)

  if (w >= 2) {
    assign("secondary_mean", secondary_mean, envir = .GlobalEnv)
    assign("secondary_angle_difference", secondary_angle_difference, envir = .GlobalEnv)
    assign("secondary_mix", secondary_mix, envir = .GlobalEnv)
    assign("secondary_mean_season_start", secondary_mean_season_start, envir = .GlobalEnv)
    assign("secondary_mean_season_end", secondary_mean_season_end, envir = .GlobalEnv)
    assign("count_within_range2", count_within_range2, envir = .GlobalEnv)
  }

  if (w == 3) {
    assign("tertiary_mean", tertiary_mean, envir = .GlobalEnv)
    assign("tertiary_angle_difference", tertiary_angle_difference, envir = .GlobalEnv)
    assign("tertiary_mix", tertiary_mix, envir = .GlobalEnv)
    assign("tertiary_mean_season_start", tertiary_mean_season_start, envir = .GlobalEnv)
    assign("tertiary_mean_season_end", tertiary_mean_season_end, envir = .GlobalEnv)
    assign("count_within_range3", count_within_range3, envir = .GlobalEnv)
  }
}

  analyze_components <- function(w, proportion, mean_directions_degrees_normalized, maxima, minima, correlation, circular_angles) {

    find_nearest_circular <- function(target, points, direction = "before") {
      circular_differences <- abs((points - target + 180) %% 360 - 180)

      if (direction == "before") {
        valid_indices <- which((target - points + 360) %% 360 > 0 & (target - points + 360) %% 360 < 180)
        if (length(valid_indices) == 0) {
          valid_indices <- which(points == max(points))
        }
      } else {
        valid_indices <- which((points - target + 360) %% 360 > 0 & (points - target + 360) %% 360 < 180)
        if (length(valid_indices) == 0) {
          valid_indices <- which(points == min(points))
        }
      }
      nearest_index <- valid_indices[which.min(circular_differences[valid_indices])]
      return(points[nearest_index])
    }

    is_within_range <- function(angle, start, end) {
      if (start < end) {
        angle >= start & angle <= end
      } else {
        angle >= start | angle <= end
      }
    }

    angles_degrees <- circular_angles * (180 / pi)

    if (w == 1) {
      primary_mix <- max(proportion)
      rsquared_value <- correlation
      primary_mean <- mean_directions_degrees_normalized
      primary_mean_season_start <- maxima
      primary_mean_season_end <- minima

      count_within_range1 <- sum(is_within_range(angles_degrees, primary_mean_season_start, primary_mean_season_end))
      print(paste("count_within_range:", count_within_range1))

      primary_angle_difference <- primary_mean_season_end - primary_mean_season_start
      if (primary_angle_difference < 0) primary_angle_difference <- primary_angle_difference + 360
      print(primary_angle_difference)
      export_component_variables( w = w,  primary_mean = primary_mean, primary_angle_difference = primary_angle_difference,
      rsquared_value = rsquared_value,primary_mix = primary_mix, primary_mean_season_start = primary_mean_season_start,
      primary_mean_season_end = primary_mean_season_end,count_within_range1 = count_within_range1, circ_correlation = correlation
)

    } else if (w == 2) {
      rsquared_value <- correlation
      max_index <- which.max(proportion)
      min_index <- which.min(proportion)

      primary_mix <- proportion[max_index]
      secondary_mix <- proportion[min_index]
      primary_mean <- mean_directions_degrees_normalized[max_index]
      secondary_mean <- mean_directions_degrees_normalized[min_index]

      primary_mean_season_start <- find_nearest_circular(primary_mean, maxima, "before")
      primary_mean_season_end <- find_nearest_circular(primary_mean, minima, "after")

      count_within_range1 <- sum(is_within_range(angles_degrees, primary_mean_season_start, primary_mean_season_end))
      print(paste("count_within_range primary:", count_within_range1))

      primary_angle_difference <- primary_mean_season_end - primary_mean_season_start
      if (primary_angle_difference < 0) primary_angle_difference <- primary_angle_difference + 360
      print(primary_angle_difference)

      secondary_mean_season_start <- find_nearest_circular(secondary_mean, maxima, "before")
      secondary_mean_season_end <- find_nearest_circular(secondary_mean, minima, "after")

      count_within_range2 <- sum(is_within_range(angles_degrees, secondary_mean_season_start, secondary_mean_season_end))
      print(paste("count_within_range secondary:", count_within_range2))

      secondary_angle_difference <- secondary_mean_season_end - secondary_mean_season_start
      if (secondary_angle_difference < 0) secondary_angle_difference <- secondary_angle_difference + 360
      print(secondary_angle_difference)
      export_component_variables(
      w = w,
      primary_mean = primary_mean,
      secondary_mean = secondary_mean,
      primary_angle_difference = primary_angle_difference,
      secondary_angle_difference = secondary_angle_difference,
      rsquared_value = rsquared_value,
      primary_mix = primary_mix,
      secondary_mix = secondary_mix,
      primary_mean_season_start = primary_mean_season_start,
      primary_mean_season_end = primary_mean_season_end,
      secondary_mean_season_start = secondary_mean_season_start,
      secondary_mean_season_end = secondary_mean_season_end,
      count_within_range1 = count_within_range1,
      count_within_range2 = count_within_range2,
      circ_correlation = correlation  )

    } else if (w == 3) {
      rsquared_value <- correlation
      sorted_indices <- order(proportion, decreasing = TRUE)
      max_index <- sorted_indices[1]
      second_highest_index <- sorted_indices[2]
      min_index <- sorted_indices[3]

      primary_mean <- mean_directions_degrees_normalized[max_index]
      secondary_mean <- mean_directions_degrees_normalized[second_highest_index]
      tertiary_mean <- mean_directions_degrees_normalized[min_index]

      primary_mix <- proportion[max_index]
      secondary_mix <- proportion[second_highest_index]
      tertiary_mix <- proportion[min_index]

      primary_mean_season_start <- find_nearest_circular(primary_mean, maxima, "before")
      primary_mean_season_end <- find_nearest_circular(primary_mean, minima, "after")

      count_within_range1 <- sum(is_within_range(angles_degrees, primary_mean_season_start, primary_mean_season_end))
      print(paste("count_within_range primary:", count_within_range1))

      primary_angle_difference <- primary_mean_season_end - primary_mean_season_start
      if (primary_angle_difference < 0) primary_angle_difference <- primary_angle_difference + 360
      print(primary_angle_difference)

      secondary_mean_season_start <- find_nearest_circular(secondary_mean, maxima, "before")
      secondary_mean_season_end <- find_nearest_circular(secondary_mean, minima, "after")

      count_within_range2 <- sum(is_within_range(angles_degrees, secondary_mean_season_start, secondary_mean_season_end))
      print(paste("count_within_range secondary:", count_within_range2))

      secondary_angle_difference <- secondary_mean_season_end - secondary_mean_season_start
      if (secondary_angle_difference < 0) secondary_angle_difference <- secondary_angle_difference + 360
      print(secondary_angle_difference)

      tertiary_mean_season_start <- find_nearest_circular(tertiary_mean, maxima, "before")
      tertiary_mean_season_end <- find_nearest_circular(tertiary_mean, minima, "after")

      count_within_range3 <- sum(is_within_range(angles_degrees, tertiary_mean_season_start, tertiary_mean_season_end))
      print(paste("count_within_range tertiary:", count_within_range3))

      tertiary_angle_difference <- tertiary_mean_season_end - tertiary_mean_season_start
      if (tertiary_angle_difference < 0) tertiary_angle_difference <- tertiary_angle_difference + 360
      print(tertiary_angle_difference)
      export_component_variables(
      w = w,
      primary_mean = primary_mean,
      secondary_mean = secondary_mean,
      tertiary_mean = tertiary_mean,
      primary_angle_difference = primary_angle_difference,
      secondary_angle_difference = secondary_angle_difference,
      tertiary_angle_difference = tertiary_angle_difference,
      rsquared_value = rsquared_value,
      primary_mix = primary_mix,
      secondary_mix = secondary_mix,
      tertiary_mix = tertiary_mix,
      primary_mean_season_start = primary_mean_season_start,
      primary_mean_season_end = primary_mean_season_end,
      secondary_mean_season_start = secondary_mean_season_start,
      secondary_mean_season_end = secondary_mean_season_end,
      tertiary_mean_season_start = tertiary_mean_season_start,
      tertiary_mean_season_end = tertiary_mean_season_end,
      count_within_range1 = count_within_range1,
      count_within_range2 = count_within_range2,
      count_within_range3 = count_within_range3,
      circ_correlation = correlation)

    } else {
      stop("Unexpected number of components w.")
    }
  }

  # Call analyze_components AFTER all computations
  analyze_components(
    w = best_model_index,
    proportion = mixing_proportion,
    mean_directions_degrees_normalized = conversion$degrees,
    maxima = maxima_info$angle,
    minima = minima_info$angle,
    correlation = circular_correlation,
    circular_angles = angles_circular
  )
}
''')

[1] 1
theta:
      [,1]      [,2]
1 2.538791 0.7915087
alpha:
[1] 1
L:
[1] 89.34162
[1] "Circular Correlation Coefficient: 0.916589292464587"
[1] "Estimated mean directions (in radians):"
       1 
0.302216 
[1] "Mean directions in degrees (normalized):"
      1 
17.3157 
     angle  density first_derivative
1 343.3647 2.439772         3.623505
     angle  density first_derivative
1 51.27425 2.439296        -3.623506
[1] "count_within_range: 63"
[1] 67.90958


<rpy2.rinterface_lib.sexp.NULLType object at 0x14e27f499e10> [0]

<span style="font-size:36px; font-weight:bold; color:black;">Step 10: Value Extraction</span>

In [ ]:
def r_scalar(name):
    if name in robjects.globalenv:
        return float(np.array(robjects.globalenv[name])[0])
    return None

In [ ]:
def get_event_radians(start_deg, end_deg):
    start_rad = math.radians(start_deg)
    end_rad = math.radians(end_deg)
    if start_rad <= end_rad:
        return angles[(angles >= start_rad) & (angles <= end_rad)]
    else:
        return angles[(angles >= start_rad) | (angles <= end_rad)]

In [ ]:
def compute_surge_stats(filtered_rads):
    surge_dict = {}
    for rad in filtered_rads:
        values = top_peaks_df[top_peaks_df['doy radian'] == rad]
        if not values.empty:
            surge_dict[rad] = values['filtered_surge'].values
    surge_values = [v for sublist in surge_dict.values() for v in sublist]
    if len(surge_values) == 0:
        return None, None, None
    surge_array = np.array(surge_values)
    return surge_array.mean(), surge_array.std(), (surge_array.std() / surge_array.mean()) * 100


In [ ]:
# Gather R outputs
results = {
    'uniformity_test': r_scalar('uniformitytest'),
    'optimal_season_number': r_scalar('best_model_index'),
    'rsquared_value': r_scalar('rsquared_value'),
    'circ_correlation': r_scalar('circ_correlation'),
    'primary season': {
        'mean_direction': r_scalar('primary_mean'),
        'start': r_scalar('primary_mean_season_start'),
        'end': r_scalar('primary_mean_season_end'),
        'length': r_scalar('primary_angle_difference'),
        'mix': r_scalar('primary_mix'),
        'surge_count': r_scalar('count_within_range1')
    },
    'secondary season': {
        'mean_direction': r_scalar('secondary_mean'),
        'start': r_scalar('secondary_mean_season_start'),
        'end': r_scalar('secondary_mean_season_end'),
        'length': r_scalar('secondary_angle_difference'),
        'mix': r_scalar('secondary_mix'),
        'surge_count': r_scalar('count_within_range2')
    },
    'tertiary season': {
        'mean_direction': r_scalar('Tertiary_mean'),
        'start': r_scalar('Tertiary_mean_season_start'),
        'end': r_scalar('Tertiary_mean_season_end'),
        'length': r_scalar('Tertiary_angle_difference'),
        'mix': r_scalar('Tertiary_mix'),
        'surge_count': r_scalar('count_within_range3')
    }
}

In [ ]:
# Process each season if it exists
for season in ['primary season', 'secondary season', 'tertiary season']:
    season_data = results.get(season)
    if season_data and season_data.get('start') is not None and season_data.get('end') is not None:
        filtered_rads = get_event_radians(season_data['start'], season_data['end'])
        mean_surge, std_surge, cv_surge = compute_surge_stats(filtered_rads)
        season_data['surge_mean'] = mean_surge
        season_data['surge_std'] = std_surge
        season_data['surge_cv'] = cv_surge
    else:
        print(f"No {season.capitalize()} found for this location.")

No Secondary season found for this location.
No Tertiary season found for this location.


In [ ]:
# Final results dict
import pprint
pprint.pprint(results,sort_dicts=False)

{'uniformity_test': 1.0,
 'optimal_season_number': 1.0,
 'rsquared_value': 0.9165892924645869,
 'circ_correlation': 0.9165892924645869,
 'primary season': {'mean_direction': 17.315699950881832,
                    'start': 343.3646729345869,
                    'end': 51.274254850970195,
                    'length': 67.90958191638327,
                    'mix': 1.0,
                    'surge_count': 63.0,
                    'surge_mean': np.float64(0.6215656510128603),
                    'surge_std': np.float64(0.08949579600957426),
                    'surge_cv': np.float64(14.398446224262571)},
 'secondary season': {'mean_direction': None,
                      'start': None,
                      'end': None,
                      'length': None,
                      'mix': None,
                      'surge_count': None},
 'tertiary season': {'mean_direction': None,
                     'start': None,
                     'end': None,
                     'length': None,
     